# Documentation Helper - Ingestion Pipeline (Colab)

This notebook ports the ingestion workflow from `ingestion.py` in the upstream project.

It will:
- crawl LangChain docs with Tavily
- convert pages to LangChain `Document` objects
- split into chunks
- embed and store into Chroma (`chroma_db`)


In [1]:
# Colab/local install cell
%pip -q install   certifi   python-dotenv   chromadb   langchain-core   langchain-classic   langchain-chroma   langchain-openai   langchain-pinecone   langchain-tavily   nest_asyncio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587

In [3]:
import asyncio
import os
import ssl
from typing import List

import certifi
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_tavily import TavilyCrawl, TavilyExtract, TavilyMap
import nest_asyncio

In [4]:
class Colors:
    PURPLE = "\033[95m"
    CYAN = "\033[96m"
    DARKCYAN = "\033[36m"
    BLUE = "\033[94m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    BOLD = "\033[1m"
    UNDERLINE = "\033[4m"
    END = "\033[0m"


def log_info(message: str, color: str = Colors.CYAN):
    print(f"{color}ℹ️  {message}{Colors.END}")


def log_success(message: str):
    print(f"{Colors.GREEN}✅ {message}{Colors.END}")


def log_error(message: str):
    print(f"{Colors.RED}❌ {message}{Colors.END}")


def log_warning(message: str):
    print(f"{Colors.YELLOW}⚠️  {message}{Colors.END}")


def log_header(message: str):
    print(f"\n{Colors.BOLD}{Colors.PURPLE}{'='*60}{Colors.END}")
    print(f"{Colors.BOLD}{Colors.PURPLE}🚀 {message}{Colors.END}")
    print(f"{Colors.BOLD}{Colors.PURPLE}{'='*60}{Colors.END}\n")

In [6]:
# Put your .env file next to this notebook in Colab/local runtime
load_dotenv('.env', override=True)

ssl_context = ssl.create_default_context(cafile=certifi.where())
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

if not os.getenv('OPENAI_API_KEY'):
    raise ValueError('Missing OPENAI_API_KEY in .env')
if not os.getenv('TAVILY_API_KEY'):
    raise ValueError('Missing TAVILY_API_KEY in .env')

In [7]:
embeddings = OpenAIEmbeddings(
    model='text-embedding-3-small',
    show_progress_bar=False,
    chunk_size=50,
    retry_min_seconds=10
)

In [8]:
vectorstore = Chroma(
    persist_directory='chroma_db',
    embedding_function=embeddings
)

In [9]:
tavily_extract = TavilyExtract()
tavily_map = TavilyMap(max_depth=5, max_breadth=20, max_pages=1000)
tavily_crawl = TavilyCrawl()

nest_asyncio.apply()

In [10]:
async def index_documents_async(documents: List[Document], batch_size: int = 50):
    """Process documents in batches asynchronously."""
    log_header('VECTOR STORAGE PHASE')
    log_info(
        f'📚 VectorStore Indexing: Preparing to add {len(documents)} documents to vector store',
        Colors.DARKCYAN,
    )

    batches = [documents[i : i + batch_size] for i in range(0, len(documents), batch_size)]
    log_info(
        f'📦 VectorStore Indexing: Split into {len(batches)} batches of {batch_size} documents each'
    )

    async def add_batch(batch: List[Document], batch_num: int):
        try:
            await vectorstore.aadd_documents(batch)
            log_success(
                f'VectorStore Indexing: Successfully added batch {batch_num}/{len(batches)} ({len(batch)} documents)'
            )
        except Exception as e:
            log_error(f'VectorStore Indexing: Failed to add batch {batch_num} - {e}')
            return False
        return True

    tasks = [add_batch(batch, i + 1) for i, batch in enumerate(batches)]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    successful = sum(1 for result in results if result is True)

    if successful == len(batches):
        log_success(
            f'VectorStore Indexing: All batches processed successfully! ({successful}/{len(batches)})'
        )
    else:
        log_warning(
            f'VectorStore Indexing: Processed {successful}/{len(batches)} batches successfully'
        )


In [11]:
async def main():
    """Main async function to orchestrate the entire process."""
    log_header('DOCUMENTATION INGESTION PIPELINE')

    log_info('🗺️  TavilyCrawl: Starting to crawl the documentation site', Colors.PURPLE)
    res = tavily_crawl.invoke(
        {
            'url': 'https://python.langchain.com/',
            'max_depth': 2,
            'extract_depth': 'advanced',
        }
    )

    all_docs = []
    for tavily_crawl_result_item in res.get('results', []):
        url = tavily_crawl_result_item.get('url', '')
        raw_content = tavily_crawl_result_item.get('raw_content', '')
        log_info(f'TavilyCrawl: Successfully crawled {url} from documentation site')
        all_docs.append(
            Document(
                page_content=raw_content,
                metadata={'source': url},
            )
        )

    log_header('DOCUMENT CHUNKING PHASE')
    log_info(
        f'✂️  Text Splitter: Processing {len(all_docs)} documents with 4000 chunk size and 200 overlap',
        Colors.YELLOW,
    )
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=200)
    splitted_docs = text_splitter.split_documents(all_docs)
    log_success(
        f'Text Splitter: Created {len(splitted_docs)} chunks from {len(all_docs)} documents'
    )

    await index_documents_async(splitted_docs, batch_size=500)

    log_header('PIPELINE COMPLETE')
    log_success('🎉 Documentation ingestion pipeline finished successfully!')
    log_info('📊 Summary:', Colors.BOLD)
    log_info(f'   • Documents extracted: {len(all_docs)}')
    log_info(f'   • Chunks created: {len(splitted_docs)}')

    return all_docs, splitted_docs


In [12]:
all_docs, splitted_docs = await main()



🚀 DOCUMENTATION INGESTION PIPELINE

ℹ️  🗺️  TavilyCrawl: Starting to crawl the documentation site
ℹ️  TavilyCrawl: Successfully crawled https://python.langchain.com/ from documentation site
ℹ️  TavilyCrawl: Successfully crawled https://docs.langchain.com/ from documentation site
ℹ️  TavilyCrawl: Successfully crawled https://academy.langchain.com/ from documentation site
ℹ️  TavilyCrawl: Successfully crawled https://blog.langchain.com/ from documentation site
ℹ️  TavilyCrawl: Successfully crawled https://trust.langchain.com/ from documentation site
ℹ️  TavilyCrawl: Successfully crawled https://forum.langchain.com/ from documentation site
ℹ️  TavilyCrawl: Successfully crawled https://changelog.langchain.com/ from documentation site
ℹ️  TavilyCrawl: Successfully crawled https://smith.langchain.com/ from documentation site
ℹ️  TavilyCrawl: Successfully crawled https://www.langchain.com/ from documentation site
ℹ️  TavilyCrawl: Successfully crawled https://langchain.com/about from document